<a href="https://colab.research.google.com/github/Hardik-Jaluthariya/multi-agent-researcher/blob/main/Multiagent_Research_ASS_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community google-generativeai langchain-google-genai chromadb sentence-transformers tiktoken duckduckgo-search requests beautifulsoup4 streamlit


INFO: pip is looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 28.3 MB

In [ ]:
# ==== Gemini LLM + Embeddings Initialization (Crisp Version) ====
import os, logging
from getpass import getpass

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ---- API Key ----
API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass("Paste your GOOGLE_API_KEY (hidden): ").strip()
if not API_KEY:
    raise RuntimeError("No GOOGLE_API_KEY provided.")
os.environ["GOOGLE_API_KEY"] = API_KEY

# ---- Imports ----
try:
    from langchain_google_genai import GoogleGenerativeAI, GoogleGenerativeAIEmbeddings
except ImportError as e:
    logger.error("Install langchain_google_genai: %s", e)
    raise

# ---- Use exact models ----
LLM_MODEL = "gemini-2.5-flash"
EMBED_MODEL = "models/gemini-embedding-001"

# ---- Initialize LLM ----
llm = GoogleGenerativeAI(model=LLM_MODEL, temperature=0.0, google_api_key=API_KEY)
logger.info("LLM initialized: %s", LLM_MODEL)

# ---- Initialize Embeddings ----
embeddings = GoogleGenerativeAIEmbeddings(model=EMBED_MODEL, google_api_key=API_KEY)
_vec = embeddings.embed_query("test probe")
logger.info("Embeddings ready: %s (dim=%d)", EMBED_MODEL, len(_vec))

# ---- Optional LLM Smoke Test ----
try:
    resp = llm("Summarize in one sentence: Transformers are models that use self-attention.")
    logger.info("LLM test call succeeded. Preview: %s", str(resp)[:400])
except Exception as e:
    logger.warning("LLM test call failed: %s", e)
VSTORE_DIR = 'vstore_chroma'
print(f"\n--- FINAL INITIALIZATION SUCCESS ---\nLLM Model Used: {LLM_MODEL}\nEmbeddings Model Used: {EMBED_MODEL}")


Paste your GOOGLE_API_KEY (hidden): ··········


/tmp/ipython-input-2731255092.py:36: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  resp = llm("Summarize in one sentence: Transformers are models that use self-attention.")



--- FINAL INITIALIZATION SUCCESS ---
LLM Model Used: gemini-2.5-flash
Embeddings Model Used: models/gemini-embedding-001


In [ ]:
from typing import List
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.schema import Document

# initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)

def load_and_split_pdf(path: str) -> List[dict]:
    loader = PyPDFLoader(path)
    raw_docs = loader.load()
    docs = []
    for d in raw_docs:
        chunks = text_splitter.split_text(d.page_content)
        for c in chunks:
            docs.append({"page_content": c, "metadata": {"source": d.metadata.get('source', path)}})
    return docs

def index_documents(docs: List[dict], persist_directory: str = VSTORE_DIR):
    # create Chroma index
    chroma = Chroma.from_documents(
        [Document(page_content=d['page_content'], metadata=d['metadata']) for d in docs],
        embeddings,
        persist_directory=persist_directory
    )
    chroma.persist()
    return chroma

# simple retriever helper
def get_retriever(chroma_index, k=4):
    return chroma_index.as_retriever(search_kwargs={"k": k})


In [ ]:
# --------------- Planner (PydanticOutputParser) ---------------
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain.chains import LLMChain
from langchain.schema.runnable import RunnableLambda
from pydantic import BaseModel, Field
from typing import List


# ---- Planner Schema ----
class PlannerSchema(BaseModel):
    subquestions: List[str] = Field(..., description="List of concise sub-questions (4-8)")
    intents: List[str] = Field(None, description="(optional) intent per subquestion e.g. literature, code, experiment")


planner_parser = PydanticOutputParser(pydantic_object=PlannerSchema)

# 🪄 Get the format instructions manually
format_instructions = planner_parser.get_format_instructions()
fmt_escaped = format_instructions.replace("{", "{{").replace("}", "}}")


planner_prompt = PromptTemplate(
    input_variables=["research_question"],

    template=f"""You are a research planner.
Given the research question below, RETURN a JSON object that matches the following schema:
{fmt_escaped}

Research question: {{research_question}}
"""
)


planner_chain = LLMChain(llm=llm, prompt=planner_prompt)

# Example:
# out = planner_chain.invoke({"research_question": "How does retrieval-augmented chain-of-thought improve model accuracy in scientific domains?"})
# parsed = planner_parser.parse(out["text"])
# print(parsed.subquestions)


# --------------- Router (RunnableLambda) ---------------
def router_fn(subq_with_intent: dict):
    subq = subq_with_intent.get('subquestion', '').lower()
    intent = subq_with_intent.get('intent') or ''
    if 'code' in intent or 'code' in subq or 'experiment' in intent or 'run' in subq or 'compute' in subq:
        return code_chain
    if 'survey' in intent or 'statist' in subq or 'experiment' in intent:
        return experiment_chain
    return literature_chain


router = RunnableLambda(func=router_fn)


# --------------- Analyst (structured CoT style) ---------------
class AnalystSchema(BaseModel):
    answer: str
    steps: List[str] = Field(..., description='Concise numbered reasoning steps')
    confidence: str = Field(None, description='Optional model confidence')

analyst_parser = PydanticOutputParser(pydantic_object=AnalystSchema)
format_instructions_analyst = analyst_parser.get_format_instructions()
fmy_escaped = format_instructions_analyst.replace("{", "{{").replace("}", "}}")

analyst_prompt = PromptTemplate(
    input_variables=['subquestion', 'context_docs'],
    template=f"""You are an analyst.
Answer the subquestion using the provided context.
Return a JSON object matching this schema:
{fmy_escaped}

Subquestion: {{subquestion}}

Context:
{{context_docs}}
"""
)

analyst_chain = LLMChain(llm=llm, prompt=analyst_prompt)


# --------------- Verifier (structured check) ---------------
class VerifierSchema(BaseModel):
    result: str
    notes: str

verifier_parser = PydanticOutputParser(pydantic_object=VerifierSchema)
format_instructions_verifier = verifier_parser.get_format_instructions()
fmu_escaped = format_instructions_verifier.replace("{", "{{").replace("}", "}}")

verifier_prompt = PromptTemplate(
    input_variables=['claim_summary', 'evidence_docs'],
    template=f"""You are a verifier.
Check whether the following claim summary is supported by the evidence.
Return a JSON object matching the schema:
{fmu_escaped}
Claims: {{claim_summary}}

Evidence:
{{evidence_docs}}
"""
)

verifier_chain = LLMChain(llm=llm, prompt=verifier_prompt)

# Define chains for the router
literature_chain = analyst_chain
experiment_chain = analyst_chain
code_chain = analyst_chain # Placeholder for now

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose file via UI
pdf_path = list(uploaded.keys())[0]
print("Uploaded:", pdf_path)


Saving Resume 2024.pdf to Resume 2024.pdf
Uploaded: Resume 2024.pdf


In [ ]:
docs = load_and_split_pdf(pdf_path)


In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 6.7 MB/s eta 0:00:00


In [ ]:
# Example research question
research_question = "What are the main applications of retrieval-augmented generation (RAG) in scientific research?"

# Step 1: Planner makes subquestions
plan = planner_chain.invoke({"research_question": research_question})
parsed_plan = planner_parser.parse(plan["text"])

print("🧩 Generated Subquestions:")
for i, q in enumerate(parsed_plan.subquestions, 1):
    print(f"{i}. {q}")

# Step 2: For each subquestion, use router + analyst + verifier mock flow
for subq in parsed_plan.subquestions:
    print("\n" + "-"*80)
    print(f"🧠 Subquestion: {subq}")

    # Mock some fake context — later, you’ll extract text from the uploaded PDF
    context = "The paper discusses how RAG improves retrieval and generation synergy in multiple scientific domains."

    # Router decides which chain to use
    chosen_chain = router_fn({'subquestion': subq, 'intent': None})
    print(f"⚙️ Routed to: {chosen_chain}")

    # Analyst step
    analyst_out = chosen_chain.invoke({"subquestion": subq, "context_docs": context})
    parsed_analyst = analyst_parser.parse(analyst_out["text"])
    print("🧩 Analyst Reasoning Steps:", parsed_analyst.steps)
    print("💡 Analyst Answer:", parsed_analyst.answer)

    # Verifier step
    verifier_out = verifier_chain.invoke({"claim_summary": parsed_analyst.answer, "evidence_docs": context})
    parsed_verifier = verifier_parser.parse(verifier_out["text"])
    print("✅ Verifier Result:", parsed_verifier.result)
    print("📝 Notes:", parsed_verifier.notes)


🧩 Generated Subquestions:
1. How does RAG enhance literature review and knowledge synthesis in scientific research?
2. What are RAG's applications in generating hypotheses and designing experiments?
3. How is RAG utilized for data interpretation and contextualization in scientific studies?
4. Can RAG assist in scientific writing, such as drafting papers or grant proposals?
5. What role does RAG play in facilitating peer review and scientific critique?
6. In which specific scientific disciplines (e.g., biomedicine, materials science) is RAG most impactful?
7. What are the current limitations and challenges of deploying RAG in scientific research environments?
8. How does RAG compare to traditional information retrieval or generative AI methods in scientific applications?

--------------------------------------------------------------------------------
🧠 Subquestion: How does RAG enhance literature review and knowledge synthesis in scientific research?
⚙️ Routed to: verbose=False prompt=

KeyboardInterrupt: 

In [ ]:
# ---------- 1) Ensure uploaded file is present ----------
# (You said you already ran files.upload(); pdf_path holds the filename)
print("Using uploaded PDF path:", pdf_path)

# ---------- 2) Load & split the PDF into chunks ----------
print("Loading & splitting PDF ...")
docs_dicts = load_and_split_pdf(pdf_path)
print(f"Number of chunks created: {len(docs_dicts)}")

# ---------- 3) Index into Chroma (persist) ----------
# Use your VSTORE_DIR variable. You can change persist_directory if you want other location.
print("Indexing into Chroma (this may take a while for large docs)...")
from langchain.schema import Document
docs_for_index = [Document(page_content=d['page_content'], metadata=d['metadata']) for d in docs_dicts]

# create/overwrite index (this uses your index_documents function which already persists)
# use the dicts returned by your loader/splitter
chroma_index = index_documents(docs_dicts, persist_directory=VSTORE_DIR)

print("Indexing complete. Persisted to:", VSTORE_DIR)

# ---------- 4) Build retriever ----------
retriever = get_retriever(chroma_index, k=4)
print("Retriever ready (k=4).")

# ---------- Helper: safe retriever -> context builder ----------
def build_context_for_query(query, k=4):
    docs = retriever.get_relevant_documents(query)
    # Show debug snippets
    print(f"Retrieved {len(docs)} chunks for query: {query!r}")
    for i, d in enumerate(docs):
        src = d.metadata.get("source", "unknown")
        snippet = d.page_content[:400].replace("\n", " ")
        print(f"  [{i+1}] source={src} snippet={snippet}...")
    context = "\n\n".join([d.page_content for d in docs])
    return context, docs

# ---------- 5) Use Planner to generate subquestions ----------
research_question = "Provide a short analysis of this document and propose 5 subquestions that cover its main claims and open problems."
print("\nRunning planner...")
# If your planner_prompt used escaped format instructions (fmt_escaped), calling is simple:
plan_out = planner_chain.invoke({"research_question": research_question})
raw_plan = plan_out.get("text") if isinstance(plan_out, dict) else plan_out
print("RAW planner output:\n", raw_plan[:1000], "\n")

# Parse planner output robustly
try:
    planner_parsed = planner_parser.parse(raw_plan if isinstance(raw_plan, str) else raw_plan.get("text",""))
    planned = []
    for i, subq in enumerate(planner_parsed.subquestions):
        intent = planner_parsed.intents[i] if (planner_parsed.intents and i < len(planner_parsed.intents)) else None
        planned.append({"subquestion": subq, "intent": intent})
    print("Planned subquestions:", planned)
except Exception as e:
    print("Planner parse error:", e)
    print("Planner raw output (helpful for debugging):\n", raw_plan)
    # fallback: treat the entire document as one subquestion
    planned = [{"subquestion": research_question, "intent": "literature"}]

# ---------- 6) Run combined loop: Router -> Retrieve -> Analyst -> Verifier ----------
results = []
for item in planned:
    subq = item["subquestion"]
    intent = item.get("intent") or ""
    print("\n" + "="*60)
    print("Subquestion:", subq)
    print("Intent:", intent)

    # choose chain using python router function (avoid router.invoke validation)
    chosen_chain = router_fn({"subquestion": subq, "intent": intent})
    print("Routed to:", repr(chosen_chain))

    # retrieve context from chroma BEFORE invoking chain
    context_docs, retrieved_docs = build_context_for_query(subq, k=4)

    # decide which chain to call (if placeholder, fallback to analyst_chain)
    chain_to_call = chosen_chain if getattr(chosen_chain, "invoke", None) else analyst_chain

    # Prepare inputs - include format_instructions if chain expects it
    chain_inputs = {"subquestion": subq, "context_docs": context_docs}
    # some of your prompt variants expect format_instructions injected; include if needed
    if getattr(chain_to_call, "prompt", None):
        ivars = getattr(chain_to_call.prompt, "input_variables", [])
        if "format_instructions_analyst" in ivars:
            chain_inputs["format_instructions_analyst"] = analyst_parser.get_format_instructions()
        if "format_instructions" in ivars:
            # planner style (unlikely here), but safe to include
            chain_inputs["format_instructions"] = planner_parser.get_format_instructions()

    # invoke the chain
    print("Invoking chosen chain...")
    out = chain_to_call.invoke(chain_inputs)
    raw = out.get("text") if isinstance(out, dict) else out
    print("RAW ANALYST OUTPUT:\n", raw)

    # parse the analyst output
    try:
        parsed_ans = analyst_parser.parse(raw if isinstance(raw, str) else raw.get("text",""))
        claim = parsed_ans.answer
        steps = parsed_ans.steps
        confidence = parsed_ans.confidence
    except Exception as e:
        print("Analyst parsing failed:", e)
        # show raw for debugging
        claim = raw if isinstance(raw, str) else str(raw)
        steps = []
        confidence = None

    # verifier run
    ver_inputs = {"claim_summary": claim, "evidence_docs": context_docs}
    ver_out = verifier_chain.invoke(ver_inputs)
    ver_raw = ver_out.get("text") if isinstance(ver_out, dict) else ver_out
    print("RAW VERIFIER OUTPUT:\n", ver_raw)

    try:
        ver_parsed = verifier_parser.parse(ver_raw if isinstance(ver_raw, str) else ver_raw.get("text",""))
        verdict = ver_parsed.result
        notes = ver_parsed.notes
    except Exception as e:
        print("Verifier parsing failed:", e)
        verdict = None
        notes = ver_raw

    results.append({
        "subquestion": subq,
        "intent": intent,
        "routed_to": repr(chosen_chain),
        "answer": claim,
        "steps": steps,
        "confidence": confidence,
        "verdict": verdict,
        "verifier_notes": notes,
        "retrieved_sources": [d.metadata.get("source") for d in retrieved_docs]
    })

# ---------- 7) Show aggregated results ----------
print("\n\n=== AGGREGATED RESULTS ===")
for r in results:
    print("-"*80)
    print("Subquestion:", r["subquestion"])
    print("Routed to:", r["routed_to"])
    print("Answer:", r["answer"])
    print("Steps:", r["steps"])
    print("Verifier verdict:", r["verdict"])
    print("Verifier notes:", r["verifier_notes"])
    print("Sources:", r["retrieved_sources"])


Using uploaded PDF path: Resume 2024.pdf
Loading & splitting PDF ...
Number of chunks created: 6
Indexing into Chroma (this may take a while for large docs)...


/tmp/ipython-input-1991029632.py:27: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  chroma.persist()


Indexing complete. Persisted to: vstore_chroma
Retriever ready (k=4).

Running planner...
RAW planner output:
 ```json
{
  "subquestions": [
    "How will the \"document\" be provided or accessed for analysis?",
    "What specific criteria define a \"short analysis\" in terms of depth and content?",
    "What methodology should be employed to accurately identify the document's \"main claims\"?",
    "What methodology should be employed to accurately identify the document's \"open problems\"?",
    "What guidelines should be followed to formulate the 5 proposed subquestions?",
    "How should the proposed subquestions ensure comprehensive coverage of both claims and open problems?"
  ]
}
``` 

Planned subquestions: [{'subquestion': 'How will the "document" be provided or accessed for analysis?', 'intent': None}, {'subquestion': 'What specific criteria define a "short analysis" in terms of depth and content?', 'intent': None}, {'subquestion': 'What methodology should be employed to accur

/tmp/ipython-input-3596654493.py:28: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)


RAW ANALYST OUTPUT:
 ```json
{
 "answer": "The provided context does not contain information on how the document will be provided or accessed for analysis. The context itself is the document being analyzed.",
 "steps": [
  "Review the provided context for any explicit mentions of how the document is supplied or accessed.",
  "Identify that the context is a resume-like document and does not include metadata or instructions regarding its provision or access method.",
  "Conclude that the information requested by the subquestion is not present in the given context."
 ]
}
```
RAW VERIFIER OUTPUT:
 ```json
{
 "result": "Supported",
 "notes": "The evidence provided is a document (a resume). It does not contain any information about how this document was provided or how it should be accessed for analysis. The claim accurately states that the context *is* the document being analyzed, and that it lacks information on its provision or access methods."
}
```

Subquestion: What specific criteria d

In [ ]:
# cell 5
# Define your retriever here before calling run_multi_agent_pipeline
# Example:
# chroma_index = index_documents(your_loaded_docs, embeddings, persist_directory=VSTORE_DIR)
# retriever = get_retriever(chroma_index)


def run_multi_agent_pipeline(question: str, retriever, top_k_per_subq=4):
    results = []
    logger.info('Planner: generating sub-questions...')
    raw = planner_chain.invoke({'research_question': question})
    parsed = planner_parser.parse(raw['text'])
    subqs = parsed.subquestions

    # Add a check to ensure subqs is not empty
    if not subqs:
        logger.warning("Planner returned no subquestions. Cannot proceed.")
        return {'question': question, 'final_summary': "Could not generate subquestions.", 'detailed': []}

    for i, subq in enumerate(subqs):
        logger.info(f'Processing sub-question {i+1}/{len(subqs)}: {subq}')

        # Assuming intents are available, otherwise set a default or infer
        intent = parsed.intents[i] if parsed.intents and len(parsed.intents) > i else ""
        logger.info(f"Inferred intent: {intent or 'None'}")

        # This line requires 'retriever' to be defined before calling this function
        docs = retriever.get_relevant_documents(subq)
        context_text = """
---
""".join([f"[source:{getattr(d,'metadata',{}).get('source','unknown')}]\n{d.page_content[:1200]}" for d in docs])


        # route to appropriate chain (router_fn should be defined elsewhere)
        # For this example, we'll use the analyst_chain directly.
        # chosen_chain = router.invoke({'subquestion': subq, 'intent': intent})
        chosen_chain = analyst_chain


        # run analyst via chosen chain (which should follow analyst schema)
        out = chosen_chain.invoke({'subquestion': subq, 'context_docs': context_text})
        parsed_answer = analyst_parser.parse(out['text'])


        # verification
        evidence_text = """
""".join([f"{d.page_content[:1200]}" for d in docs])
        raw_ver = verifier_chain.invoke({'claim_summary': parsed_answer.answer, 'evidence_docs': evidence_text})
        parsed_ver = verifier_parser.parse(raw_ver['text'])


        results.append({
            'subquestion': subq,
            'answer': parsed_answer.answer,
            'steps': parsed_answer.steps,
            'verification': parsed_ver.dict(),
            'sources': [getattr(d,'metadata',{}).get('source','unknown') for d in docs]
        })


    # final aggregation (use a summarizer chain)
    final_prompt = PromptTemplate(
        input_variables=['full_answers','orig_question'],
        template=(
            "You are an expert summarizer. Given the sub-question answers, synthesize a concise final answer. Include caveats from verifiers.\n\n"
            "Orig: {orig_question}\n\n"
            "Answers:\n{full_answers}\n\n"
            "Final summary:"
        )
    )
    final_chain = LLMChain(llm=llm, prompt=final_prompt)
    all_answers_text = """


""".join([f"Q: {r['subquestion']}\nA: {r['answer']}" for r in results])
    final_summary = final_chain.invoke({'full_answers': all_answers_text, 'orig_question': question})['text']


    return {'question': question, 'final_summary': final_summary, 'detailed': results}

# Example of how to call the function (you'll need to define your question and retriever)
# my_question = "What is the impact of climate change?"
# try:
#     pipeline_results = run_multi_agent_pipeline(my_question, retriever)
#     print("\nPipeline Final Summary:")
#     print(pipeline_results['final_summary'])
# except NameError:
#     print("\nError: 'retriever' is not defined. Please define the retriever before running the pipeline.")

In [ ]:
# cell 6
# PythonREPLTool (fast and convenient but run with caution)
from langchain_experimental.tools.python.tool import PythonREPLTool
python_tool = PythonREPLTool()
# Use the tool inside code_chain: code_chain should be an agent/chain that can call python_tool


# LangChain Sandbox (Pyodide): safer for untrusted code. See repo + examples:
# https://github.com/langchain-ai/langchain-sandbox
# You can wire the sandbox runner as a callable in your runnable pipeline.

In [ ]:
!pip install langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 4.1 MB/s eta 0:00:00
